In [1]:
from datasets import Dataset
import pandas as pd 

c:\Users\Shamshuddeen\Documents\GitHub\ML-2-Real-Time-Content-Moderation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
toxic_train = pd.read_csv(r'..\Data\toxic\train.csv')
print(type(toxic_train.columns))
toxic_train.sample(5)

<class 'pandas.Index'>


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
35182,5df5316fad149bdb,"Bill, we also have a policy called WP:BOLD. Yo...",0,0,0,0,0,0
143772,02894f4fa54fce84,"""\n\nHi again Bo [will just use the abbreviati...",0,0,0,0,0,0
64963,add974b5cf63ee89,"activities with a free man was beheaded, he wh...",0,0,0,0,0,0
47359,7e84941285b69fab,""", 25 April 2014 (UTC)\nSupport because a myth...",0,0,0,0,0,0
52025,8b3f22f84c08de36,Spelling! \n\nThanks for the advice! it was ap...,0,0,0,0,0,0


In [5]:
label_cols = toxic_train.columns.difference(["id","comment_text"])
label_cols = list(label_cols)

In [6]:
toxic_train["toxic"] = toxic_train[label_cols].max(axis=1) 

In [7]:
jigsaw_train = toxic_train[["comment_text","toxic"]].rename(columns={"comment_text":"text"})
jigsaw_train.sample()

,text,toxic
131659,It is apparent to me that you are happy to wri...,0


In [8]:
toxic_test = pd.read_csv(r'..\Data\toxic\test.csv')
test_labels = pd.read_csv(r'..\Data\toxic\test_labels.csv')
test =  toxic_test.merge(test_labels,on="id")
test = test[test["toxic"]!=-1]
test["toxic"] = test[label_cols].max(axis=1)
jigsaw_test = test[["comment_text","toxic"]].rename(columns={"comment_text":"text"})


In [9]:
jigsaw_test.sample(5)

,text,toxic
50666,== hey == \n\n hey man.. \n\n omg iam bye f...,0
85172,::Thank you very much for clearing this up.,0
100439,"I last played Ultima V on my Commodore 64, so ...",0
147059,")|here]], and [[Wikipedia:Articles for deletio...",0
39952,:You may be interested in WP:DR.,0


In [10]:
hate_meta = pd.read_csv(r'..\Data\hate\annotations_metadata.csv')
hate_meta.sample(5)

,file_id,user_id,subforum_id,num_contexts,label
10356,14113130_3,600252,1381,0,noHate
4371,31775702_3,577351,1363,0,noHate
10407,14408032_1,578693,1375,0,noHate
1225,13379549_3,575537,1345,0,noHate
7646,30954921_1,572067,1346,0,noHate


In [11]:
hate_meta["label"] =hate_meta['label'].isin(["hate"])
hate_meta.sample(5)

,file_id,user_id,subforum_id,num_contexts,label
1773,14414086_2,581032,1375,0,False
4806,14034302_1,575734,1387,0,True
2252,12848217_15,575810,1346,0,False
8413,30398448_2,573365,1359,0,False
8056,13863902_1,574905,1391,0,False


In [12]:
hate_meta["toxic"] = hate_meta['label'].astype(int)
hate_meta.sample(5)

,file_id,user_id,subforum_id,num_contexts,label,toxic
2446,13587460_1,586725,1393,0,False,0
3042,30474323_2,581520,1354,0,False,0
7332,14677022_1,614560,1371,0,False,0
5395,30712850_4,575645,1354,0,False,0
7452,30521895_1,624832,1362,0,True,1


In [13]:
texts=[]
for fid in hate_meta['file_id']:
  with open(f"../Data/hate/all_files/{fid}.txt",encoding="utf-8",errors="replace") as f:
    texts.append(f.read().strip())
hate_meta["text"] = texts
hate = hate_meta[["text","toxic"]]
hate.sample(5)

,text,toxic
2121,Y' all git your happy ass down here !!!,0
7036,My grandfather told me of england when he was ...,0
10627,Sharpless 249 and the Jellyfish Nebula Image C...,0
8775,Fee hours in any direction and there is none o...,0
828,Now we have `` judeo-christian '' faux mega-ch...,0


In [14]:
combined = pd.concat([jigsaw_test,jigsaw_train,hate],ignore_index=True)
combined.sample(5)

,text,toxic
160167,"""\nReferences get really repetitive after a wh...",0
83885,I believe your examination will yield the same...,0
211762,You a joker. You point of view is absolutely n...,0
52823,== Rodney Anoa'i Protection == \n Just noticed...,1
177296,"""What is your definition of """"edit-warring?"""" ...",0


In [15]:
dataset = Dataset.from_pandas(combined)
dataset = dataset.shuffle(seed=45)
dataset = dataset.train_test_split(test_size=0.2,seed=42)
dataset.save_to_disk(r"..\Data\combined_dataset")

Saving the dataset (1/1 shards): 100%|██████████| 46899/46899 [00:00<00:00, 355368.07 examples/s]
